[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/spinning_top/spinning_top.ipynb)

# Spinning Top

This example simulates a rigid spinning top precessing and sliding on a ground surface. Each part of the top is modelled as a solid ellipsoid, and the ground is modelled as the solid region below a paraboloid; released upright with a slight tilt, the spinning top swings its symmetry axis steadily around the vertical instead of falling over while air drag gradually slows its spin and sliding friction holds or damps its tip where it meets the ground.

Every solid shape is represented as a quadric duality map from points to planes whose pairing `quadric(point) & point` is negative inside the solid, so a part's lowest point against the ground normal is the polar image `surface.inverse()(central_plane)` of the conjugate plane through its centre. Joining six mass points per ellipsoid with their velocities under a body twist via `(mass_points * point_masses).regressive(mass_points.commutator(Rate))` sums directly into the rigid-body inertia without separate centre-of-mass or parallel-axis bookkeeping. After each unconstrained Lie-group step, contact projection applies sequential single-line impulses weighted by each contact line's rotational and translational compliance `twists & body_lines`.

In matrix Newton–Euler notation, assembling the composite inertia requires shifting each ellipsoid's $3 \times 3$ principal moment tensor $I_k$ to the centre of mass $\mathbf c$ via $I = \sum_k \bigl(R_k I_k R_k^{\mathsf T} + m_k\bigl(\|\mathbf r_k - \mathbf c\|^2 I_3 - (\mathbf r_k - \mathbf c)(\mathbf r_k - \mathbf c)^{\mathsf T}\bigr)\bigr)$, while each contact impulse splits into coupled translational and rotational updates $\Delta \mathbf v = m^{-1} j\,\mathbf n$ and $\Delta \boldsymbol\omega = I^{-1}((\mathbf p - \mathbf c) \times j\,\mathbf n)$ across an effective-mass denominator $m^{-1} + \mathbf n^{\mathsf T}\bigl[((\mathbf p - \mathbf c) \times \mathbf n)^{\mathsf T} I^{-1}((\mathbf p - \mathbf c) \times \mathbf n)\bigr]$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator

import jax
import jax.numpy as jnp
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from examples.mechanics import lie_integrators as lie
from examples.mechanics.spinning_top import render, scenarios
from examples.util.animation import save_figure, show_states
from examples.util.panels import image_figure

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Point = ga.gatype.antivector()                                                 # Antivector-
Plane = ga.gatype.vector()                                                     # Vector+
Direction = Point.derive.degenerate                                            # Antivector-
Rate = ga.gatype.bivector()                                                    # Bivector+
Forque = ga.gatype.antibivector()                                              # Antibivector-
Inertia = ga.gatype((Forque, Rate))                                            # Forque- <- Rate+
InverseInertia = ga.gatype((Rate, Forque))                                     # Rate+ <- Forque-
Motor = ga.gatype.rotor()                                                      # Rotor+
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
w = mv.w                                                                       # Plane+
axes = mv(Plane.derive.nondegenerate, np.eye(3))                               # [3] Plane+
directions = axes.dual()                                                       # [3] Antivector-
origin = w.dual()                                                              # [] Point-
up = directions[2]                                                             # [] Antivector-

## Ellipsoid parts and composite inertia

Each top combines a heavy disc, a slender stem, and a rounded tip as three solid ellipsoids centred along the symmetry axis, while the ground is modelled as the solid below a paraboloid of zero curvature. Placing six equal mass points along each ellipsoid's principal axes reproduces its exact mass, centroid, and second moments, and joining each mass point with its velocity under a rate sums directly into the composite body inertia.

In [ ]:
n_tops = scenarios.n_tops
disc_semi_axes = scenarios.disc_semi_axes
stem_semi_axes = scenarios.stem_semi_axes
tip_height = scenarios.tip_height
part_heights = scenarios.part_heights
stem_mass, tip_mass = scenarios.stem_mass, scenarios.tip_mass
tip_radii = scenarios.tip_radii
disc_masses = scenarios.disc_masses
ground_curvature = scenarios.ground_curvature
view, view_centre, view_extent, n_pixels = scenarios.view, scenarios.view_centre, scenarios.view_extent, scenarios.n_pixels

In [ ]:
def ellipsoid(centres: Point, semi_axes: np.ndarray) -> Quadric:
    """Solid ellipsoids centred on `centres` with semi-axes `semi_axes` (`[..., 3]`) form quadric maps (`[...] Plane+ <- Point-`)."""
    planes = axes - w * (axes & centres[..., None])                            # [..., 3] Plane+
    return (planes * (planes & Point) / semi_axes**2).sum(axis=-1) - w * (w & Point)


def bowl(curvature: float) -> Quadric:
    """The ground below a paraboloid of height `curvature` times the squared radial distance forms a quadric (`[] Plane+ <- Point-`)."""
    return 0.5 * (axes[2] * (w & Point) + w * (axes[2] & Point)) - curvature * (axes[:2] * (axes[:2] & Point)).sum()

In [ ]:
def top_body(part_heights: np.ndarray, semi_axes: np.ndarray, part_masses: np.ndarray) -> tuple[Quadric, Inertia, np.ndarray]:
    """Six mass points per solid ellipsoid place the parts about their centre of mass (`[..., 3] Plane+ <- Point-`) and sum their momenta into the inertia (`[...] Forque- <- Rate+`)."""
    centre_height = (part_heights * part_masses).sum(axis=-1) / part_masses.sum(axis=-1)
    centres = origin + up * (part_heights - centre_height[..., None])          # [..., 3] Point-
    # Six mass points per part along its principal axes match the solid ellipsoid's mass, centroid, and second moments.
    offsets = directions * (semi_axes * np.sqrt(3 / 5))                        # [..., 3, 3] Antivector-
    mass_points = concatenate([centres[..., None] + offsets, centres[..., None] - offsets], axis=-1)  # [..., 3, 6] Point-
    # Joining each mass point with its velocity under a rate gives its momentum map; summing yields the body inertia.
    momenta = (mass_points * (part_masses[..., None] / 6)).regressive(mass_points.commutator(Rate))   # [..., 3, 6] Forque- <- Rate+
    return ellipsoid(centres, semi_axes), momenta.sum(axis=-1).sum(axis=-1), centre_height


tip_semi_axes = np.stack(np.broadcast_arrays(tip_radii, tip_radii, tip_height), axis=-1)             # [n_tops, 3]
semi_axes = np.stack(np.broadcast_arrays(disc_semi_axes, stem_semi_axes, tip_semi_axes), axis=-2)    # [n_tops, 3, 3]
part_masses = np.stack(np.broadcast_arrays(disc_masses, stem_mass, tip_mass), axis=-1)               # [n_tops, 3]
parts, inertia, centre_height = top_body(part_heights, semi_axes, part_masses)  # [n_tops, 3] Plane+ <- Point-, [n_tops] Forque- <- Rate+
ground = bowl(ground_curvature)                                                 # [] Plane+ <- Point-

## Polar contact points and release pose

Evaluating the ground quadric at any point returns its polar plane, whose outward normal points toward the air where the quadratic form increases. Inverting an ellipsoid quadric sends planes to their poles, so the central plane perpendicular to the ground normal maps to the conjugate direction along which the ellipsoid reaches its lowest tangent point; dividing the ground's quadratic form there by twice its gradient length gives the first-order penetration depth.

In [ ]:
tilt, clearance = scenarios.tilt, scenarios.clearance

In [ ]:
def ground_normal(ground: Quadric, points: Point) -> Direction:
    """Each point's polar plane under `ground` gives the outward unit normal direction toward the air (`[...] Antivector-`)."""
    return ground(points).dual().cast(Direction) / ground(points).norm()


def lowest_point(surface: Quadric, normal: Direction) -> Point:
    """The pole of each quadric's tangent plane facing against `normal` locates its lowest point (`[...] Point-`)."""
    # Inverting the quadric maps planes to their poles; the pole of the plane at infinity is the quadric's centre.
    dual_quadric = surface.inverse()                                           # [...] Point- <- Plane+
    centre = dual_quadric(w) / (w & dual_quadric(w))                           # [...] Point-
    # The polar of the central plane perpendicular to `normal` is the conjugate direction toward the tangent point.
    central_plane = normal.dual() - w * (normal.dual() & centre)               # [...] Plane+
    conjugate = dual_quadric(central_plane)                                    # [...] Antivector-
    return centre + conjugate / (-(central_plane & conjugate) * (w & dual_quadric(w))).square_root()

In [ ]:
def contact_state(motor: Motor, parts: Quadric, ground: Quadric) -> tuple[Point, Direction, Scalar]:
    """Each placed part's lowest point against the ground normal at its centre yields the contact point, normal, and signed penetration depth."""
    placed_parts = motor[..., None] >> parts(motor[..., None] << Point)        # [..., 3] Plane+ <- Point-
    centres = placed_parts.solve(w)                                            # [..., 3] Point-
    contact = lowest_point(placed_parts, ground_normal(ground, centres / (w & centres)))  # [..., 3] Point-
    # Dividing the ground's quadratic form by twice its gradient length measures depth to first order.
    depth = -(ground(contact) & contact) / (2 * ground(contact).norm())        # [..., 3] Scalar+
    return contact, ground_normal(ground, contact), depth


lifts = (mv.zw * ((centre_height * np.cos(tilt) + clearance) / 2)).exp()       # [n_tops] Rotor+
lean_turn = (mv.zx * (tilt / 2)).exp()                                         # [] Rotor+
starts = (lifts * lean_turn).normalized()                                      # [n_tops] Rotor+
_, _, start_depths = contact_state(starts, parts, ground)                      # [n_tops, 3] Scalar+
save_figure(image_figure(render.row(starts, parts, ground, view, view_centre, view_extent, n_pixels)), "spinning_top_start");

From left to right, the four tops represent the reference top, a sharper tip, a heavier red disc, and a duplicate of the reference top placed on slipperier ground, each lifted until its tilted tip starts just clear of the checkerboard plane.

## Sequential contact projection and precession

Each substep advances the body freely by fourth-order Lie Runge–Kutta integration under gravity and air drag, then sweeps over the three ellipsoids in sequence so every contact sees the pose left by the preceding corrections. Pressing out along the ground normal and holding against tangential slip act along single force lines through the contact point, weighted by each line's compliance with the inverse inertia so an impact far from the centre of mass turns the body more readily than one near it; dynamic friction then damps any remaining tangential slip on the implied body rate.

In [ ]:
gravity = scenarios.gravity
static_friction = scenarios.static_friction
drag = scenarios.drag
spin_rate = scenarios.spin_rate
dynamic_frictions = scenarios.dynamic_frictions
dt = scenarios.dt
n_frames = scenarios.n_frames
n_steps_per_frame = scenarios.n_steps_per_frame
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def ground_tangent(direction: Direction, normal: Direction) -> Direction:
    """Subtracting the component along `normal` projects `direction` onto the ground's tangent plane (`[...] Antivector-`)."""
    return direction - normal * (direction.dual() | normal.dual())


def clamp(value: Scalar, limit: Scalar) -> Scalar:
    """Clamping holds the magnitude of `value` at most equal to `limit` (`[...] Scalar+`)."""
    magnitude = value.abs()                                                    # [...] Scalar+
    smaller = (magnitude + limit - (magnitude - limit).abs()) / 2              # [...] Scalar+
    return value * smaller / (magnitude + 1e-12)

In [ ]:
def line_compliance(motor: Motor, inverse_inertia: InverseInertia, lines: Forque) -> tuple[Rate, Scalar]:
    """Pulling `lines` into the body frame maps unit loads to body twists (`[...] Rate+`) and scalar compliances (`[...] Scalar+`)."""
    body_lines = motor << lines                                                # [...] Forque-
    twists = inverse_inertia(body_lines)                                       # [...] Rate+
    return twists, twists & body_lines


def impulse_step(motor: Motor, twists: Rate, impulses: Scalar) -> Motor:
    """Applying `impulses` along `twists` steps `motor` by the summed rigid-body displacement (`[...] Rotor+`)."""
    step_motor = ((twists * impulses).sum(axis=-1) * -0.5).exp()               # [...] Rotor+
    return (motor * step_motor).normalized()

In [ ]:
def project_contacts(before: Motor, motor: Motor, inverse_inertia: InverseInertia, parts: Quadric, ground: Quadric,
                     static: float, dynamic: np.ndarray, dt: float) -> tuple[Motor, Rate]:
    """Sequential Gauss-Seidel impulses press each penetrating part out of `ground` and damp tangential slip within the friction cone."""
    part_inertia = inverse_inertia[..., None]                                  # [..., 1] Rate+ <- Forque-
    # Materializing in the motor's context lets `jax.lax.scan` step through the parts in sequence.
    parts = parts.materialize(motor.context)                                   # [..., ...] Plane+ <- Point-
    dynamic_limit = jnp.asarray(dynamic)[..., None] / dt

    def press_step(motor: Motor, part: Quadric) -> tuple[Motor, Scalar]:
        # Trailing singleton axes handle one contact part at a time across any leading batch of bodies.
        contact, normal, depth = contact_state(motor, part[..., None], ground)
        # Normal projection presses the contact out along the ground normal by its positive depth.
        twists, compliance = line_compliance(motor[..., None], part_inertia, contact & normal)
        press_impulse = depth.clip(0, np.inf) / compliance                     # [..., 1] Scalar+
        motor = impulse_step(motor, twists, press_impulse)                     # [...] Rotor+
        # Static friction restores the contact's tangential position when the impulse fits inside the static cone.
        previous_contact = before[..., None] >> (motor[..., None] << contact)  # [..., 1] Point-
        slip = ground_tangent(contact - previous_contact, normal)              # [..., 1] Antivector-
        slip_length = slip.dual().norm()                                       # [..., 1] Scalar+
        twists, compliance = line_compliance(motor[..., None], part_inertia, contact & (-slip / (slip_length + 1e-12)))
        stick_impulse = slip_length / (compliance + 1e-12)                     # [..., 1] Scalar+
        return impulse_step(motor, twists, stick_impulse * (stick_impulse <= press_impulse * static)), press_impulse

    motor, press_impulses = jax.lax.scan(press_step, motor, parts)
    # The relative motor from `before` to the projected `motor` determines the post-projection body rate.
    rate = (before.inverse() * motor).normalized().log() * (-2 / dt)           # [...] Rate+

    def friction_step(rate: Rate, part_and_impulse: tuple[Quadric, Scalar]) -> tuple[Rate, None]:
        part, press_impulse = part_and_impulse
        contact, normal, _ = contact_state(motor, part[..., None], ground)
        # Dynamic friction opposes the remaining tangential velocity up to `dynamic` times the normal impulse.
        sliding = ground_tangent(contact.commutator(motor[..., None] >> rate[..., None]), normal)
        speed = sliding.dual().norm()                                          # [..., 1] Scalar+
        twists, compliance = line_compliance(motor[..., None], part_inertia, contact & (-sliding / (speed + 1e-12)))
        drag_impulse = clamp(speed / (compliance + 1e-12), press_impulse * dynamic_limit)
        return rate + (twists * drag_impulse).sum(axis=-1), None

    rate, _ = jax.lax.scan(friction_step, rate, (parts, press_impulses))
    return motor, rate

In [ ]:
def spin(motor: Motor, rate: Rate, parts: Quadric, inertia: Inertia, masses: np.ndarray, ground: Quadric,
         gravity: float, drag: float, static_friction: float, dynamic_friction: np.ndarray,
         dt: float, n_frames: int, n_steps_per_frame: int) -> Iterator[tuple[Motor, Rate]]:
    """Yields the body poses and rates (`[...] Rotor+`, `[...] Rate+`) across `n_frames` frames spaced `n_steps_per_frame` steps apart."""
    inverse_inertia = inertia.inverse()                                        # [...] Rate+ <- Forque-
    # Scanning in `jax.lax.scan` moves the part axis to axis 0 so each scan iteration sees one part across all tops.
    scan_parts = stack([parts[..., index] for index in range(parts.shape[-1])])

    def external_forque(motor: Motor, rate: Rate) -> Forque:
        # Gravity acts downward through the centre of mass while air drag opposes the body's angular and linear momentum.
        weight = motor << ((motor >> origin) & (up * (-gravity * masses)))     # [...] Forque-
        return weight - inertia(rate) * drag

    def substep(state: tuple[Motor, Rate], _: None) -> tuple[tuple[Motor, Rate], None]:
        before, rate = state
        predicted, rate = lie.explicit_rk4(before, rate, inertia, inverse_inertia, dt, external_forque)
        projected, rate = project_contacts(before, predicted.normalized(), inverse_inertia, scan_parts, ground,
                                           static_friction, dynamic_friction, dt)
        return (projected.retyped(Motor), rate.retyped(Rate)), None

    frame_step: Callable = jax.jit(lambda state: jax.lax.scan(substep, state, length=n_steps_per_frame)[0])
    context = JaxContext(ga)
    state = (motor.retyped(Motor).materialize(context), rate.retyped(Rate).materialize(context))
    for _ in range(n_frames):
        yield state[0].materialize(mv.context), state[1].materialize(mv.context)
        state = frame_step(state)


def lean_angle(motor: Motor) -> Scalar:
    """The angle between the body's symmetry axis and the vertical `up` measures the top's lean (`[...] Scalar+`)."""
    return ((motor >> up).dual() | up.dual()).abs().arccos()


start_rates = (mv.xy * spin_rate).broadcast_to(starts.shape)                   # [n_tops] Rate+
state_stream = spin(starts, start_rates, parts, inertia, part_masses.sum(axis=-1),
                    ground, gravity, drag, static_friction, dynamic_frictions, dt, n_frames, n_steps_per_frame)
states = show_states(state_stream, render.row_frames, parts, ground, view, view_centre, view_extent, n_pixels,
                     name="spinning_top", duration_ms=frame_duration_ms, n_frames=n_frames);

All four tops settle onto their tips, precess steadily around the vertical without toppling, and wander differently across the checkerboard: the sharper tip drifts less than the rounder tip, the heavier disc precesses more slowly with greater gyroscopic stiffness, and the top on slipperier ground skids wider arcs before settling.

In [ ]:
# checks
# Every top starts clear of the ground, stays upright below 45 degrees, and spins in the same sense while air drag slows it.
motors = stack([frame_motors for frame_motors, _ in states])                   # [n_frames, n_tops] Rotor+
rates = stack([frame_rates for _, frame_rates in states])                      # [n_frames, n_tops] Rate+
leans = lean_angle(motors)                                                     # [n_frames, n_tops] Scalar+
spins = -(mv.xy | rates)                                                       # [n_frames, n_tops] Scalar+
assert (start_depths <= 0).all()
assert (leans < np.pi / 4).all()
assert (spins > 0).all() and (spins[-1] < spins[0]).all()